In [2]:
import sqlite3
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html
from dash.dependencies import Output, Input
import os

app = dash.Dash(__name__)
server = app.server

# Constants
COLUMN_TO_DISPLAY = "num_function_evaluations"
EVALUATION_TABLE = 'EVALUATION_EPISODES'  # Replace with actual table name
POLICIES_TABLE = 'CONSTRUCTED_POLICIES'  # Replace with actual table name

# Load table names
def get_table_names(global_database_connection):
    query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
    return pd.read_sql_query(query_tables, global_database_connection)

# Query to get all rows from the CONFIG table
def load_config_table(global_database_connection):
    query_config = "SELECT * FROM CONFIG"
    return pd.read_sql_query(query_config, global_database_connection)

# Format the CONFIG table output
def format_config_table(config_df):
    formatted_table = "CONFIG Table:\n"
    formatted_table += "| Database Path        | Key                  | Value                     |\n"
    formatted_table += "|----------------------|----------------------|---------------------------|\n"

    for index, row in config_df.iterrows():
        database_path = os.path.basename(row['database_path'])
        key = row['key']
        value = row['value']
        formatted_table += f"| {database_path:<20} | {key:<20} | {value:<25} |\n"

    return formatted_table

# Check if the table exists
def table_exists(tables, table_name):
    return table_name in tables['name'].to_list()

# Load data from the evaluation table
def load_evaluation_data(global_database_connection, column_to_display):
    query_evaluation = f"""
      SELECT policy_id,
             AVG({column_to_display}) AS avg_value,
             COUNT(*) AS row_count,
             AVG(({column_to_display} - avg_value) * ({column_to_display} - avg_value)) AS var_value
      FROM (
          SELECT policy_id,
                 {column_to_display},
                 AVG({column_to_display}) OVER (PARTITION BY policy_id) AS avg_value
          FROM {EVALUATION_TABLE}
      )
      GROUP BY policy_id
    """
    return pd.read_sql_query(query_evaluation, global_database_connection).set_index('policy_id')

# Load num_total_timesteps from the policies table
def load_policies_data(global_database_connection):
    query_policies = f"""
      SELECT policy_id, num_total_timesteps, database_path
      FROM {POLICIES_TABLE}
    """
    return pd.read_sql_query(query_policies, global_database_connection)

def do(done_database_path):
    # Connect to the SQLite database
    global_database_connection = sqlite3.connect(done_database_path)

    tables = get_table_names(global_database_connection)
    config_rows = load_config_table(global_database_connection)
    formatted_config_table = format_config_table(config_rows)

    assert table_exists(tables, EVALUATION_TABLE), f"Table '{EVALUATION_TABLE}' does not exist in the database."
    assert table_exists(tables, POLICIES_TABLE), f"Table '{POLICIES_TABLE}' does not exist in the database."

    df_evaluation = load_evaluation_data(global_database_connection, COLUMN_TO_DISPLAY)
    df_policies = load_policies_data(global_database_connection)

    # Merge the two dataframes on policy_id
    df_merged = df_evaluation.merge(df_policies.drop_duplicates(subset='policy_id'), on='policy_id', how='left')
    df_merged.set_index('policy_id', inplace=True)

    # Calculate standard deviation
    df_merged['stddev_value'] = np.sqrt(df_merged['var_value'])

    # Separate the baseline data (policy_id = -1)
    df_baseline = df_merged[df_merged.index == -1].copy()
    df_others = df_merged[df_merged.index != -1].copy()

    # Get the number of unique policies
    num_policies = df_policies['database_path'].nunique()

    # Add number of episodes per policy
    df_others['episodes_per_policy'] = df_others['row_count'] / num_policies

    # Baseline values
    if not df_baseline.empty:
        baseline_value = df_baseline['avg_value'].values[0]
        baseline_stddev = np.sqrt(df_baseline['var_value'].values[0])
        lower_bound = baseline_value - 0.65 * baseline_stddev
        upper_bound = baseline_value + 0.65 * baseline_stddev
    else:
        baseline_value = baseline_stddev = lower_bound = upper_bound = None

    # Create the line plot for other policies with conditional coloring and percentage calculation
    colors = []
    percentages = []
    for i, row in df_others.iterrows():
        if baseline_value is not None:
            percentage_diff = ((row['avg_value'] - baseline_value) / baseline_stddev) * 100
            percentages.append(percentage_diff)
            colors.append('green' if lower_bound <= row['avg_value'] <= upper_bound else '#1f77b4')
        else:
            percentages.append(None)
            colors.append('#1f77b4')

    df_others['percentage_diff'] = percentages

    fig = px.line(
        df_others,
        x='num_total_timesteps',
        y='avg_value',
        labels={'num_total_timesteps': 'Number of Total Timesteps', 'avg_value': f'Average {COLUMN_TO_DISPLAY.replace("_", " ").title()}'},
        title=f'Average {COLUMN_TO_DISPLAY.replace("_", " ").title()} by Number of Total Timesteps'
    )

    # Update the plot with hover data and colors for other policies
    fig.update_traces(
        mode='markers+lines',
        hovertemplate='<b>Number of Total Timesteps:</b> %{x}<br>' +
                      '<b>Average Value:</b> %{y}<br>' +
                      '<b>Row Count:</b> %{customdata[0]}<br>' +
                      '<b>Number of seeds:</b> ' + str(num_policies) + '<br>' +
                      '<b>Number of Episodes per seed:</b> %{customdata[1]}<br>' +
                      '<b>Standard Deviation:</b> %{customdata[2]}<br>' +
                      '<b>Percentage Difference from Baseline:</b> %{customdata[3]:.2f}%',
        marker=dict(color=colors)
    )

    # Add hover data for other policies
    fig.update_traces(customdata=df_others[['row_count', 'episodes_per_policy', 'stddev_value', 'percentage_diff']])

    # Add the standard deviation shading for other policies
    fig.add_traces([
        go.Scatter(
            x=df_others['num_total_timesteps'],
            y=df_others['avg_value'] + df_others['stddev_value'],
            mode='lines',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip'
        ),
        go.Scatter(
            x=df_others['num_total_timesteps'],
            y=df_others['avg_value'] - df_others['stddev_value'],
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor='rgba(0,100,80,0.2)',
            showlegend=False,
            hoverinfo='skip'
        )
    ])

    # Add the baseline line with hoverable points and standard deviation shading
    if not df_baseline.empty:
        fig.add_trace(
            go.Scatter(
                x=df_others['num_total_timesteps'],
                y=[baseline_value] * len(df_others),
                mode='lines',
                line=dict(color='orange', dash='dash'),
                name='Baseline',
                hoverinfo='y',
                hovertemplate='<b>Baseline:</b><br>' +
                              '<b>Average Value:</b> %{y}<br>' +
                              f'<b>Standard Deviation:</b> {baseline_stddev}'
            )
        )

        fig.add_traces([
            go.Scatter(
                x=df_others['num_total_timesteps'],
                y=[baseline_value + baseline_stddev] * len(df_others),
                mode='lines',
                line=dict(width=0),
                showlegend=False,
                hoverinfo='skip'
            ),
            go.Scatter(
                x=df_others['num_total_timesteps'],
                y=[baseline_value - baseline_stddev] * len(df_others),
                mode='lines',
                line=dict(width=0),
                fill='tonexty',
                fillcolor='rgba(255,165,0,0.2)',
                showlegend=False,
                hoverinfo='skip'
            )
        ])

    # Update layout to start y-axis from 0 and set dragmode to select
    fig.update_layout(
        yaxis=dict(range=[0, None]),
        dragmode='select'
    )

    global_database_connection.close()

    return fig, df_others, num_policies, df_baseline

def setup_callbacks(fig_id, selected_data_id, horizontal_lines_plot_id, df_others, num_policies, df_baseline, done_database_path):
    @app.callback(
        [Output(selected_data_id, 'children'),
         Output(horizontal_lines_plot_id, 'figure')],
        [Input(fig_id, 'selectedData')]
    )
    def display_selected_data(selectedData):
        if selectedData is None:
            return "No points selected", go.Figure()

        points = selectedData['points']
        selected_points = [
            {
                'x': point['x'],
                'y': point['y'],
                'customdata': point['customdata']
            }
            for point in points
        ]

        selected_points = sorted(selected_points, key=lambda point: point['x'])

        # Initialize the nested dictionary
        policy_details_dict = {}

        # Establish a new SQLite connection
        display_selected_data_database_connection = sqlite3.connect(done_database_path)

        cursor = display_selected_data_database_connection.cursor()

        # Pick an arbitrary database_path
        cursor.execute("SELECT database_path FROM POLICY_DETAILS WHERE policy_id = -1 LIMIT 1")
        arbitrary_database_path = cursor.fetchone()[0]

        # Query to fetch mapping from fitness to mutation_size for the selected database_path and policy_id -1
        cursor.execute("""
            SELECT fitness, mutation_size
            FROM POLICY_DETAILS
            WHERE policy_id = -1 AND database_path = ?
        """, (arbitrary_database_path,))

        # Fetch all rows and create a dictionary mapping fitness to mutation_size
        baseline_policy = {row[0]: row[1] for row in cursor.fetchall()}

        lines_fig = go.Figure()

        for point in points:
            num_total_timesteps = point['x']

            # Step 1: Retrieve rows from CONSTRUCTED_POLICIES where num_total_timesteps is not null
            query_policies_with_timesteps = f"""
                SELECT database_path, policy_id
                FROM CONSTRUCTED_POLICIES
                WHERE num_total_timesteps = {num_total_timesteps}
            """

            # Execute the query and load the data into a DataFrame
            df_policies_with_timesteps = pd.read_sql_query(query_policies_with_timesteps, display_selected_data_database_connection)

            # Step 2 and 3: For each unique database_path and policy_id, extract the mapping from fitness to mutation_size
            for _, row in df_policies_with_timesteps.iterrows():
                db_path_value = row['database_path']
                policy_id_value = row['policy_id']

                # Define the query to extract fitness and mutation_size from POLICY_DETAILS for the current database_path and policy_id
                query_policy_details = f"""
                    SELECT fitness, mutation_size
                    FROM POLICY_DETAILS
                    WHERE database_path = '{db_path_value}' AND policy_id = {policy_id_value}
                """

                # Execute the query and load the data into a DataFrame
                df_policy_details = pd.read_sql_query(query_policy_details, display_selected_data_database_connection)

                # Create the fitness to mutation_size mapping
                fitness_to_mutation_size = dict(zip(df_policy_details['fitness'], df_policy_details['mutation_size']))

                # Add the mapping to the nested dictionary
                if num_total_timesteps not in policy_details_dict:
                    policy_details_dict[num_total_timesteps] = {}
                if db_path_value not in policy_details_dict[num_total_timesteps]:
                    policy_details_dict[num_total_timesteps][db_path_value] = fitness_to_mutation_size

            # Draw a plotly curve for that timestep.

            # Aggregate all fitness to mutation size mappings for the current point for each policy_id
            assert num_total_timesteps in policy_details_dict
            that_timestep_policies = policy_details_dict[num_total_timesteps]
            aggregated_fitness_to_mutation_size = {}
            count_per_fitness = {}

            for db_path_value in that_timestep_policies:
                fitness_to_mutation_size = policy_details_dict[num_total_timesteps][db_path_value]
                for fitness, mutation_size in fitness_to_mutation_size.items():
                    if fitness not in aggregated_fitness_to_mutation_size:
                        aggregated_fitness_to_mutation_size[fitness] = 0
                        count_per_fitness[fitness] = 0
                    aggregated_fitness_to_mutation_size[fitness] += mutation_size
                    count_per_fitness[fitness] += 1

            # Compute the average mutation size for each fitness
            average_fitness_to_mutation_size = {fitness: aggregated_fitness_to_mutation_size[fitness] / count_per_fitness[fitness]
                                                for fitness in aggregated_fitness_to_mutation_size}

            # Create the plot with the average mapping for the current point
            lines_fig.add_trace(
                go.Scatter(
                    x=list(average_fitness_to_mutation_size.keys()),
                    y=list(average_fitness_to_mutation_size.values()),
                    mode='lines+markers',
                    name=f'Total Timesteps: {num_total_timesteps:,}',
                    hovertemplate='<b>Fitness:</b> %{x}<br><b>Mutation Size:</b> %{y}<br><b>Number of seeds:</b> %{customdata}'
                )
            )

        lines_fig.add_trace(
            go.Scatter(
                x=list(baseline_policy.keys()),
                y=list(baseline_policy.values()),
                mode='lines+markers',
                name=f'λ = √(n⋅(n−f(x)))',
                hovertemplate='<b>Fitness:</b> %{x}<br><b>Mutation Size:</b> %{y}'
            )
        )

        # Add hover data
        lines_fig.update_traces(customdata=[count_per_fitness[fitness] for fitness in average_fitness_to_mutation_size.keys()])

        # Close the new SQLite connection
        display_selected_data_database_connection.close()

        lines_fig.update_layout(
            yaxis=dict(range=[0, None]),
            title='Average Fitness to Mutation Size Mapping',
            xaxis_title='Fitness',
            yaxis_title='Mutation Size',
            legend_title='Policy ID and Total Timesteps',
        )

        return html.Pre(str(selected_points)), lines_fig

def extract_info(db_path):
    pattern = re.compile(r"_merged_\('(\d+)', '([\d.]+)'\)")
    match = pattern.search(db_path)
    if match:
        dimensionality = match.group(1)
        initial_fitness = match.group(2)

        if "reward_scaling" in db_path:
            reward_scaling_info = "WITH reward scaling as reward /= self.dimensionality"
        elif "July_28" in db_path:
            reward_scaling_info = "WITHOUT reward scaling as reward /= self.dimensionality"
        else:
            reward_scaling_info = "Unknown reward scaling"

        return f"Dimensionality: {dimensionality}, Initial fitness: {initial_fitness}, {reward_scaling_info}"
    else:
        return "No match found"

def main(database_paths):
    graphs = []
    for i, db_path in enumerate(database_paths):
        fig, df_others, num_policies, df_baseline = do(db_path)

        graphs.append(html.H2(extract_info(db_path)))
        graphs.append(dcc.Graph(id=f'line-plot{i+1}', figure=fig))
        graphs.append(html.Div(id=f'selected-data{i+1}'))
        graphs.append(dcc.Graph(id=f'horizontal-lines-plot{i+1}'))

        setup_callbacks(f'line-plot{i+1}', f'selected-data{i+1}', f'horizontal-lines-plot{i+1}', df_others, num_policies, df_baseline, db_path)

    app.layout = html.Div(graphs)

    app.run_server(debug=True, use_reloader=False, host='0.0.0.0', port=8050)

if __name__ == '__main__':
    database_paths = [
        "/home/dimitri/code/oll_onemax/computed/July_28/_merged_('100', '0.5').db",
        "/home/dimitri/code/oll_onemax/computed/reward_scaling/_merged_('100', '0.5').db",

        "/home/dimitri/code/oll_onemax/computed/July_28/_merged_('100', '0.85').db",
        "/home/dimitri/code/oll_onemax/computed/reward_scaling/_merged_('100', '0.85').db",

        "/home/dimitri/code/oll_onemax/computed/July_28/_merged_('200', '0.5').db",
        "/home/dimitri/code/oll_onemax/computed/reward_scaling/_merged_('200', '0.5').db",

        "/home/dimitri/code/oll_onemax/computed/July_28/_merged_('500', '0.5').db",
        "/home/dimitri/code/oll_onemax/computed/reward_scaling/_merged_('500', '0.5').db",
    ]
    main(database_paths)
